# Phase 3: The Novelty - Adversarial Training (GRL)
This is the research-gap solution. We use a **Gradient Reversal Layer (GRL)** to remove language bias from the multilingual embeddings, forcing the model to learn language-invariant features.

In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Function
from transformers import XLMRobertaModel

# 1. Custom Gradient Reversal Layer (GRL)
class GradientReversalLayer(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # During backward pass, gradients are reversed and scaled by alpha
        output = grad_output.neg() * ctx.alpha
        return output, None

def grad_reverse(x, alpha=1.0):
    return GradientReversalLayer.apply(x, alpha)

print("GRL Autograd Function implemented.")

## 2. Adversarial Model Architecture
The model has a shared XLM-R backbone and two branches:
1. **Fake News Classifier**: CNN + Bi-LSTM (The Task).
2. **Language Discriminator**: MLP with GRL (The Adversary).

In [ ]:
class AdversarialFakeNewsModel(nn.Module):
    def __init__(self, model_name='xlm-roberta-base', num_languages=2):
        super(AdversarialFakeNewsModel, self).__init__()
        # Shared Backbone
        self.xlm_roberta = XLMRobertaModel.from_pretrained(model_name)
        
        # --- Task Branch (Fake News Detection) ---
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=768, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), # 64*2 for Bidirectional
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # --- Adversarial Branch (Language Discriminator) ---
        self.lang_discriminator = nn.Sequential(
            nn.Linear(768, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_languages)
        )
        
    def forward(self, input_ids, attention_mask, alpha=1.0):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        pooled_output = outputs.pooler_output
        
        # 1. Fake News Classification
        cnn_out = self.cnn(sequence_output.transpose(1, 2))
        _, (hn, _) = self.lstm(cnn_out.transpose(1, 2))
        # Concat last hidden states of Bi-LSTM
        hn_cat = torch.cat((hn[0], hn[1]), dim=1)
        label_out = self.classifier(hn_cat)
        
        # 2. Language Discrimination with GRL
        reverse_features = grad_reverse(pooled_output, alpha)
        lang_logits = self.lang_discriminator(reverse_features)
        
        return label_output, lang_logits

print("Adversarial Architecture Ready.")

## 3. Advanced Dual-Loss Training
We minimize classification loss while the GRL ensures we maximize language discrimination error (un-learning language features).